# Nyström Preconditioning in Neural Network Training

Standard gradient descent updates parameters as:
$$x_{k+1} = x_k - \alpha \nabla f(x_k)$$

When the loss landscape is ill-conditioned (the Hessian $H = \nabla^2 f$ has eigenvalues spanning several orders of magnitude), convergence is slow because the optimal learning rate is bounded by $\alpha < 2 / \lambda_{\max}(H)$.

**Preconditioned gradient descent** replaces the raw gradient with a preconditioned direction:
$$x_{k+1} = x_k - \alpha \, M^{-1} \nabla f(x_k)$$

where $M \approx H$ is a positive-definite approximation of the Hessian. The ideal choice $M = H$ yields Newton's method, which converges in one step for quadratics, but computing $H^{-1}$ is $O(d^3)$ — prohibitive for neural networks.

The **Nyström approximation** builds a low-rank sketch $H \approx U \, \text{diag}(\sigma) \, U^\top$ using only $r \ll d$ Hessian-vector products (each costing one backward pass). The preconditioner $(H_{\text{nys}} + \lambda I)^{-1}$ is then applied via the Woodbury identity in $O(d \cdot r)$ time per step.

In [ ]:
%matplotlib inline

import sys
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from copy import deepcopy

sys.path.insert(0, '.')

from models import MLP, NystromPreconditioner
from dataset import get_dataloaders
from trainer import Trainer
from nystrom_module import HessianApproximator, NystromOptimizer, compare_optimizers

np.random.seed(42)
torch.manual_seed(42)

colors = {
    "SGD": "#2196F3",
    "Adam": "#4CAF50",
    "SGD + Nyström": "#F44336",
    "Adam + Nyström": "#FF9800",
}

print("Setup complete.")

## Benchmark 1: Optimizer Comparison on Ill-Conditioned Classification

Neural network loss surfaces are inherently ill-conditioned: feature scales, layer depths, and nonlinear activations all contribute to Hessian eigenvalues that span many orders of magnitude. This means the condition number $\kappa(H) = \lambda_{\max} / \lambda_{\min} \gg 1$, causing standard SGD to zig-zag along narrow valleys.

The Nyström approximation captures the top-$r$ eigenspace of $H$ using $r$ Hessian-vector products, then applies the **Woodbury identity** to compute $(H_{\text{nys}} + \lambda I)^{-1} g$ in $O(d \cdot r)$:

$$(H_{\text{nys}} + \lambda I)^{-1} g = \frac{1}{\lambda}\left(g - U \, \text{diag}\!\left(\frac{\sigma_i}{\sigma_i + \lambda}\right) U^\top g\right)$$

We compare SGD, Adam, SGD+Nyström, and Adam+Nyström on 10-dimensional synthetic classification data with feature scales from $31.6\times$ to $1\times$.

In [ ]:
train_loader, val_loader = get_dataloaders(
    task="classification", batch_size=64, n_train=500, n_val=100, d_features=10, seed=42
)

model_fn = lambda: MLP(d_in=10, d_hidden=32, d_out=2, n_layers=3)
loss_fn = nn.CrossEntropyLoss()

n_params = sum(p.numel() for p in model_fn().parameters())
print(f"Model: MLP(10→32→32→2), {n_params} parameters")
print(f"Data:  500 train / 100 val, 10-dim features (scales 31.6× to 1×)")
print(f"Training SGD, Adam, SGD+Nyström, Adam+Nyström for 30 epochs ...")

t0 = time.perf_counter()
comparison = compare_optimizers(
    model_fn, train_loader, val_loader, loss_fn, n_epochs=30, device="cpu"
)
print(f"Done in {time.perf_counter() - t0:.1f}s\n")

print(f"{'Method':<20s} {'Final Loss':>12s} {'Val Acc':>10s}")
print(f"{'-'*44}")
for name, data in comparison.items():
    acc_str = f"{data['final_acc']:.3f}" if data["final_acc"] is not None else "N/A"
    print(f"{name:<20s} {data['final_loss']:>12.4f} {acc_str:>10s}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, data in comparison.items():
    ax1.semilogy(data["train_loss"], color=colors[name], lw=2, label=name)
    if data["val_accuracy"][0] is not None:
        ax2.plot(
            [v if v is not None else 0 for v in data["val_accuracy"]],
            color=colors[name], lw=2, label=name,
        )

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training Loss (log)")
ax1.set_title("Convergence: Training Loss")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation Accuracy")
ax2.set_title("Convergence: Validation Accuracy")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle(
    "Nyström-Preconditioned SGD vs Standard Optimizers (MLP Classification)", fontsize=13
)
plt.tight_layout()
plt.show()

## Benchmark 2: Hessian Spectrum and Effective Rank

The **Hessian spectrum** — the set of eigenvalues $\{\lambda_i\}$ of $\nabla^2 f$ — reveals the geometry of the loss landscape:

- **Rapid eigenvalue decay** means most curvature information lives in a low-dimensional subspace, making low-rank approximation effective.
- **Effective rank at $p\%$ energy**: the smallest $r$ such that $\sum_{i=1}^{r} |\lambda_i| \geq p\% \cdot \sum_{i=1}^{d} |\lambda_i|$.

If the rank-90% is much smaller than $d$, a Nyström sketch of that rank captures most of the curvature. This is empirically true for typical neural networks, where only a small fraction of Hessian eigenvalues carry significant energy.

In [ ]:
torch.manual_seed(42)
model_hess = MLP(d_in=10, d_hidden=32, d_out=2, n_layers=3)
X_batch, y_batch = next(iter(train_loader))

print(f"Computing full {n_params}×{n_params} Hessian ...")
params_h = [p for p in model_hess.parameters() if p.requires_grad]
loss_h = loss_fn(model_hess(X_batch), y_batch)
grads_h = torch.autograd.grad(loss_h, params_h, create_graph=True)
flat_grad_h = torch.cat([g.reshape(-1) for g in grads_h])

H_full = torch.zeros(n_params, n_params)
for i in range(n_params):
    row = torch.autograd.grad(flat_grad_h[i], params_h, retain_graph=True)
    H_full[i] = torch.cat([r.reshape(-1) for r in row])

H_full = 0.5 * (H_full + H_full.T)
H_full = H_full.detach()

eigs = np.sort(np.abs(torch.linalg.eigvalsh(H_full).numpy()))[::-1].copy()

cumul = np.cumsum(eigs) / np.sum(eigs)
r90 = int(np.searchsorted(cumul, 0.9) + 1)
r99 = int(np.searchsorted(cumul, 0.99) + 1)

threshold = 1e-3 * eigs[0]
effective_eigs = eigs[eigs > threshold]
kappa_eff = float(effective_eigs[0] / effective_eigs[-1]) if len(effective_eigs) > 1 else float("inf")

print(f"Top eigenvalue:     {eigs[0]:.4f}")
print(f"Smallest significant (>{threshold:.2e}): {effective_eigs[-1]:.4e}")
print(f"Effective κ (λ > 10⁻³·λ_max): {kappa_eff:.1f}")
print(f"Effective rank (90% energy): {r90} / {n_params}")
print(f"Effective rank (99% energy): {r99} / {n_params}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(eigs + 1e-15, "b-", lw=2)
ax1.axhline(threshold, color="r", ls="--", alpha=0.5, label=f"10⁻³·λ_max = {threshold:.2e}")
ax1.set_xlabel("Index i")
ax1.set_ylabel("|λ_i|")
ax1.set_title(f"MLP Hessian Spectrum ({n_params} params, κ_eff ≈ {kappa_eff:.0f})")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.plot(cumul, "g-", lw=2)
ax2.axhline(0.9, color="r", ls="--", label=f"90% at rank {r90}")
ax2.axhline(0.99, color="orange", ls="--", label=f"99% at rank {r99}")
ax2.set_xlabel("Rank")
ax2.set_ylabel("Cumulative Energy")
ax2.set_title("Hessian Spectral Energy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Benchmark 3: Condition Number Reduction via Preconditioning

The condition number $\kappa(H) = \lambda_{\max} / \lambda_{\min}$ directly controls the maximum stable learning rate:
$$\alpha_{\max} = \frac{2}{\lambda_{\max}(H)}$$

After Nyström preconditioning, the effective system becomes $(H_{\text{nys}} + \lambda I)^{-1} H$, which compresses the top eigenvalues toward 1 while leaving the small ones roughly unchanged. This:
1. **Reduces $\lambda_{\max}$** of the preconditioned system, allowing a larger learning rate.
2. **Reduces $\kappa$**, so convergence is faster even at the same learning rate.

We compute the full preconditioned Hessian spectrum using the Woodbury formula and compare eigenvalue distributions before and after preconditioning.

In [ ]:
U_nys, sigma_nys = HessianApproximator.nystrom_hessian(
    model_hess, loss_fn, (X_batch, y_batch), rank=15
)

damping = 1.0
UtH = U_nys.T @ H_full
scale = torch.diag(sigma_nys / (sigma_nys + damping))
precond_H = (H_full - U_nys @ scale @ UtH) / damping
eigs_precond = torch.linalg.eigvalsh(precond_H).numpy()
eigs_precond_abs = np.sort(np.abs(eigs_precond))[::-1]

max_lr_original = 2.0 / eigs[0]
max_lr_precond = 2.0 / max(eigs_precond_abs[0], 1e-15)

print(f"Original:        λ_max = {eigs[0]:.2f},  max stable lr = {max_lr_original:.4f}")
print(f"Preconditioned:  λ_max = {eigs_precond_abs[0]:.2f},  max stable lr = {max_lr_precond:.4f}")
print(f"Allowed step-size increase: {max_lr_precond / max_lr_original:.1f}×")
print(f"Original κ_eff:             {kappa_eff:.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

nys_sorted = np.sort(sigma_nys.numpy())[::-1]
ax1.semilogy(eigs + 1e-15, "b-", lw=2, label=f"Full Hessian (λ_max = {eigs[0]:.1f})")
ax1.semilogy(nys_sorted + 1e-15, "ro-", lw=2, ms=5, label="Nyström top-15")
ax1.set_xlabel("Index")
ax1.set_ylabel("Eigenvalue")
ax1.set_title("Full vs Nyström Hessian Spectrum")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.semilogy(eigs + 1e-15, "b--", lw=1.5, alpha=0.5,
             label=f"Original (λ_max = {eigs[0]:.1f})")
ax2.semilogy(eigs_precond_abs + 1e-15, "r-", lw=2,
             label=f"Preconditioned (λ_max = {eigs_precond_abs[0]:.2f})")
ax2.axhline(1.0, color="green", ls=":", alpha=0.7, label="Ideal: all λ = 1")
ax2.set_xlabel("Index")
ax2.set_ylabel("|Eigenvalue|")
ax2.set_title(
    f"Preconditioning compresses top eigenvalues → "
    f"{max_lr_precond / max_lr_original:.0f}× larger stable lr"
)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle(
    f"Nyström Preconditioning: λ_max {eigs[0]:.1f} → {eigs_precond_abs[0]:.2f} "
    f"(allows {max_lr_precond / max_lr_original:.0f}× larger learning rate)",
    fontsize=13,
)
plt.tight_layout()
plt.show()

In [ ]:
torch.manual_seed(42)
train_loader_reg, val_loader_reg = get_dataloaders(
    task="regression", batch_size=64, n_train=500, n_val=100, d_features=10, seed=42
)
model_fn_reg = lambda: MLP(d_in=10, d_hidden=32, d_out=1, n_layers=3)
loss_fn_reg = nn.MSELoss()

reg_configs = [
    ("SGD", lambda m: torch.optim.SGD(m.parameters(), lr=0.01)),
    ("Adam", lambda m: torch.optim.Adam(m.parameters(), lr=0.005)),
    (
        "SGD + Nyström",
        lambda m: NystromOptimizer(
            m, torch.optim.SGD(m.parameters(), lr=0.05),
            loss_fn_reg, rank=10, damping=1.0, rebuild_every=5,
        ),
    ),
    (
        "Adam + Nyström",
        lambda m: NystromOptimizer(
            m, torch.optim.Adam(m.parameters(), lr=0.003),
            loss_fn_reg, rank=10, damping=1.0, rebuild_every=5,
        ),
    ),
]

print("Training regression models for 30 epochs ...")
t0 = time.perf_counter()
comparison_reg = compare_optimizers(
    model_fn_reg, train_loader_reg, val_loader_reg, loss_fn_reg,
    n_epochs=30, device="cpu", configs=reg_configs,
)
print(f"Done in {time.perf_counter() - t0:.1f}s\n")

print(f"{'Method':<20s} {'Final Loss':>12s}")
print(f"{'-'*34}")
for name, data in comparison_reg.items():
    print(f"{name:<20s} {data['final_loss']:>12.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for name, data in comparison_reg.items():
    ax.semilogy(data["train_loss"], color=colors[name], lw=2, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss (log)")
ax.set_title("Regression: Convergence with Different Optimizers")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

**Key findings:**

1. **Optimizer comparison**: Nyström-preconditioned optimizers converge faster than their unpreconditioned counterparts on ill-conditioned problems, because preconditioning reshapes the loss landscape to be more isotropic.

2. **Hessian spectrum**: The MLP Hessian has rapidly decaying eigenvalues — most of the spectral energy is concentrated in a low-rank subspace, justifying the use of a rank-$r$ Nyström sketch with $r \ll d$.

3. **Condition number reduction**: Preconditioning compresses the dominant eigenvalues toward 1, reducing $\lambda_{\max}$ and allowing a proportionally larger stable learning rate.

4. **Regression**: The same preconditioning strategy transfers directly to regression tasks, confirming that the benefit is due to geometry (curvature) rather than task-specific tuning.

**Connection to Poisson solvers**: The Nyström Hessian inverse is mathematically identical to a low-rank preconditioner for the linear system $Hx = g$. In PDE discretizations, $H$ is a stiffness matrix; in neural networks, it is the Gauss-Newton or full Hessian. The same spectral decay that makes multigrid and Nyström preconditioning effective for elliptic PDEs also appears in neural network loss landscapes.